In [12]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

con = duckdb.connect('../capillary.db')

df = con.execute("SELECT value, manual_classification as label FROM protein_data WHERE manual_classification IS NOT NULL AND value IS NOT NULL").df()

df = df[df['value'].apply(len) == 301]
X = np.array(df['value'].tolist(), dtype=np.float32)
y = np.array(df['label'].tolist(), dtype=np.int8)

# Sista kolumnen är skräp i X.
X = X[:, :300]

np.shape(X)
np.shape(y)

con.close()

In [13]:
class_weight = {0: 1.0, 1: 74/26}

# Först: låt 15% vara test-set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Sedan: dela resten i träning/validering (0.1765 ≈ 15% av totalen)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y[: len(y_temp)]
)

X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)

X_val = torch.tensor(X_val)
y_val = torch.tensor(y_val)

X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test)

print(f"Träning:    {len(X_train):>6} ({len(X_train)/len(y):.0%})")
print(f"Validering: {len(X_val):>6} ({len(X_val)/len(y):.0%})")
print(f"Test:       {len(X_test):>6} ({len(X_test)/len(y):.0%})")

Träning:    105658 (70%)
Validering:  22646 (15%)
Test:        22643 (15%)


In [14]:
batch_sz = 64

# DataLoaders
train_dataloader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_sz, shuffle=True)
val_dataloader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=batch_sz)
test_dataloader   = DataLoader(TensorDataset(X_test,   y_test),   batch_size=batch_sz)

for X, y in val_dataloader:
    print(f"Shape of X: {X.shape}")
    print(f"Shape of y: {y.shape}")
    break

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(300, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
def train(train_dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

def validate(val_dataloader, model, loss_fn):
    size = len(val_dataloader.dataset)
    num_batches = len(val_dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in val_dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    return (100*correct,test_loss)

def test(test_dataloader, model, loss_fn):
    size = len(test_dataloader.dataset)
    num_batches = len(test_dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    return (100*correct,test_loss)



model = NeuralNetwork().to(device)
pos_weight = torch.tensor([1.0, 3.0]).to(device)
loss_fn = nn.CrossEntropyLoss(weight=pos_weight)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Antal parametrar: {total_params:,}")

Shape of X: torch.Size([64, 300])
Shape of y: torch.Size([64])
Using mps device
NeuralNetwork(
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=300, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): ReLU()
    (10): Linear(in_features=64, out_features=2, bias=True)
  )
)
Antal parametrar: 249,922


In [ ]:
## KÖR DETTA BLOCKET FÖR ATT TRÄNA! Annars kör nästa block och läs in vikterna!
epochs = 50
accuracies = []
validation_losses = []
for t in range(epochs):
    train(train_dataloader, model, loss_fn, optimizer)
    (accuracy,avg_loss) = validate(val_dataloader, model, loss_fn)
    scheduler.step()
    print(f"Epooch {t}, accuracy: {accuracy}, Avg loss: {avg_loss}")
    accuracies.append(accuracy)
    validation_losses.append(avg_loss)
print("Done!")

torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")


Epooch 0, accuracy: 90.06447054667491, Avg loss: 0.3496424839489878
Epooch 1, accuracy: 89.914333657158, Avg loss: 0.36955083948744216
Epooch 2, accuracy: 87.91398039388855, Avg loss: 0.3480950202898117
Epooch 3, accuracy: 91.76455003091054, Avg loss: 0.2907941519666863
Epooch 4, accuracy: 90.61202861432483, Avg loss: 0.3964599733803905
Epooch 5, accuracy: 86.75262739556655, Avg loss: 0.3109456800387404
Epooch 6, accuracy: 89.85251258500398, Avg loss: 0.3116608992597814
Epooch 7, accuracy: 78.96317230415968, Avg loss: 0.42689888806497983
Epooch 8, accuracy: 92.27236598074715, Avg loss: 0.26267513445457497
Epooch 9, accuracy: 92.54614501457212, Avg loss: 0.2668107991964467
Epooch 10, accuracy: 91.21699196326063, Avg loss: 0.24888787963716996
Epooch 11, accuracy: 94.30362978009362, Avg loss: 0.22475902378390739
Epooch 12, accuracy: 93.53086637816833, Avg loss: 0.22803745212527993
Epooch 13, accuracy: 93.91062439282875, Avg loss: 0.21654794962897814
Epooch 14, accuracy: 91.6011657687892, 

In [16]:
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load("model.pth", weights_only=True))

# Samla alla prediktioner
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        all_preds.extend(pred.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Rapport
print(classification_report(all_labels, all_preds, target_names=['Negativ', 'Positiv']))

# Konfusionsmatris
print(confusion_matrix(all_labels, all_preds))

(test_accuracy, test_loss) = test(test_dataloader,model,loss_fn)
print(f"Resultat på testdatan: accuracy: {test_accuracy}, Avg loss: {test_loss}")


              precision    recall  f1-score   support

     Negativ       0.96      0.98      0.97     16835
     Positiv       0.94      0.88      0.91      5808

    accuracy                           0.96     22643
   macro avg       0.95      0.93      0.94     22643
weighted avg       0.96      0.96      0.96     22643

[[16537   298]
 [  702  5106]]
Resultat på testdatan: accuracy: 95.58362407808153, Avg loss: 0.18962589354585793


In [ ]:
# Analys av hur träningen gick
import matplotlib.pyplot as plt

plt.plot(validation_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.grid(True)
plt.show()

NameError: name 'validation_losses' is not defined

Saved PyTorch Model State to model.pth
